# E04 — anomaly detection one-class su video termici

Questa baseline apprende **soltanto frame normali**. Un backbone congelato estrae feature locali; la distanza dalla memoria normale produce anomaly score e heatmap. Le box CVAT vengono usate solo alla fine per valutare la localizzazione.

In [ ]:
import os
import zipfile
from pathlib import Path
from google.colab import files

os.chdir('/content')
DATA_ROOT = Path('/content/anomaly_pilot_v3')

if not DATA_ROOT.exists():
    print('Carica anomaly_pilot_v3_colab.zip')
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1:
        raise RuntimeError('Caricare un solo file ZIP del dataset E04')
    with zipfile.ZipFile(Path('/content') / zip_names[0]) as archive:
        archive.extractall('/content')

assert (DATA_ROOT / 'manifest.csv').is_file()
assert (DATA_ROOT / 'ground_truth_boxes.csv').is_file()
print('Dataset pronto:', DATA_ROOT)

In [ ]:
import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

assert torch.cuda.is_available(), 'Attivare la GPU in Colab'
device = torch.device('cuda')

manifest = pd.read_csv(DATA_ROOT / 'manifest.csv')
gt_boxes = pd.read_csv(DATA_ROOT / 'ground_truth_boxes.csv')
display(manifest.groupby(['split', 'label']).size().rename('frame'))
print('Box CVAT di test:', len(gt_boxes))
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

weights = ResNet18_Weights.DEFAULT
resnet = resnet18(weights=weights)
# Fino a layer3: feature locali, non classificazione di oggetti.
encoder = nn.Sequential(*list(resnet.children())[:-3]).eval().to(device)
for parameter in encoder.parameters():
    parameter.requires_grad_(False)

preprocess = transforms.Compose([
    transforms.Resize((224, 280)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

class FrameDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows.reset_index(drop=True)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        image = Image.open(DATA_ROOT / row.image_path).convert('RGB')
        return preprocess(image), index

@torch.inference_mode()
def encode(images):
    features = encoder(images.to(device, non_blocking=True))
    return F.normalize(features, dim=1)

print('Backbone pronto: ResNet18 congelata, output fino a layer3')

In [ ]:
# Costruzione della memoria usando esclusivamente train/normal.
train_rows = manifest.query("split == 'train' and label == 'normal'")
train_loader = DataLoader(
    FrameDataset(train_rows), batch_size=16, shuffle=False,
    num_workers=2, pin_memory=True,
)

patch_batches = []
for images, _ in tqdm(train_loader, desc='Feature normali'):
    features = encode(images)
    patches_batch = features.permute(0, 2, 3, 1).reshape(-1, features.shape[1])
    patch_batches.append(patches_batch.cpu())

normal_memory = torch.cat(patch_batches)
maximum_memory_patches = 30000
if len(normal_memory) > maximum_memory_patches:
    generator = torch.Generator().manual_seed(SEED)
    selected = torch.randperm(len(normal_memory), generator=generator)[:maximum_memory_patches]
    normal_memory = normal_memory[selected]

normal_memory = normal_memory.to(device)
torch.save(normal_memory.half().cpu(), '/content/E04_normal_memory_fp16.pt')
print('Patch normali in memoria:', len(normal_memory))

In [ ]:
@torch.inference_mode()
def score_rows(frame_rows, description):
    dataset = FrameDataset(frame_rows)
    loader = DataLoader(
        dataset, batch_size=8, shuffle=False,
        num_workers=2, pin_memory=True,
    )
    result_rows = []
    maps = {}

    for images, indices in tqdm(loader, desc=description):
        features = encode(images)
        batch, channels, grid_h, grid_w = features.shape
        patch_vectors = features.permute(0, 2, 3, 1).reshape(-1, channels)

        minimum_distances = []
        for start in range(0, len(patch_vectors), 1024):
            distances = torch.cdist(
                patch_vectors[start:start + 1024], normal_memory
            )
            minimum_distances.append(distances.min(dim=1).values)
        patch_scores = torch.cat(minimum_distances).reshape(batch, grid_h, grid_w)
        patch_scores = patch_scores.cpu().numpy()

        for item, dataset_index in enumerate(indices.tolist()):
            row = dataset.rows.iloc[dataset_index]
            heatmap = patch_scores[item]
            key = (str(row.video), int(row.frame))
            maps[key] = heatmap
            result_rows.append({
                'image_path': row.image_path,
                'split': row.split,
                'label': row.label,
                'video': row.video,
                'frame': int(row.frame),
                'timestamp_seconds': float(row.timestamp_seconds),
                'anomaly_score': float(np.quantile(heatmap, 0.99)),
            })
    return pd.DataFrame(result_rows), maps

validation_rows = manifest.query("split == 'validation' and label == 'normal'")
test_rows = manifest.query("split == 'test' and label == 'anomaly'")
validation_scores, validation_maps = score_rows(validation_rows, 'Validation normale')
test_scores, test_maps = score_rows(test_rows, 'Test anomalo')
scores = pd.concat([validation_scores, test_scores], ignore_index=True)
all_maps = {**validation_maps, **test_maps}
print('Frame valutati:', len(scores))

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)

# La soglia usa soltanto validation normale, non i frame anomali.
threshold = float(np.quantile(validation_scores.anomaly_score, 0.95))
scores['ground_truth'] = (scores.label == 'anomaly').astype(int)
scores['predicted_anomaly'] = (scores.anomaly_score > threshold).astype(int)

precision, recall, f1, _ = precision_recall_fscore_support(
    scores.ground_truth,
    scores.predicted_anomaly,
    average='binary',
    zero_division=0,
)
tn, fp, fn, tp = confusion_matrix(
    scores.ground_truth, scores.predicted_anomaly, labels=[0, 1]
).ravel()

# Pointing game: il massimo della heatmap cade in una box CVAT?
boxes_by_frame = {
    (str(video), int(frame)): group
    for (video, frame), group in gt_boxes.groupby(['video', 'frame'])
}
pointing_hits = []
for row in test_scores.itertuples(index=False):
    key = (str(row.video), int(row.frame))
    heatmap = all_maps[key]
    grid_y, grid_x = np.unravel_index(np.argmax(heatmap), heatmap.shape)
    image = Image.open(DATA_ROOT / row.image_path)
    width, height = image.size
    point_x = (grid_x + 0.5) * width / heatmap.shape[1]
    point_y = (grid_y + 0.5) * height / heatmap.shape[0]
    frame_boxes = boxes_by_frame.get(key)
    hit = False if frame_boxes is None else any(
        box.xtl <= point_x <= box.xbr and box.ytl <= point_y <= box.ybr
        for box in frame_boxes.itertuples(index=False)
    )
    pointing_hits.append(int(hit))

summary = {
    'experiment_id': 'E04_one_class_patch_feature_baseline',
    'task': 'anomaly_detection_and_localization',
    'method': 'nearest-neighbor patch features, PatchCore-style baseline',
    'training_anomaly_labels_used': False,
    'threshold_normal_validation_p95': threshold,
    'frame_auroc': float(roc_auc_score(scores.ground_truth, scores.anomaly_score)),
    'frame_average_precision': float(average_precision_score(scores.ground_truth, scores.anomaly_score)),
    'precision': float(precision),
    'recall': float(recall),
    'f1': float(f1),
    'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    'pointing_game_accuracy': float(np.mean(pointing_hits)),
    'normal_validation_frames': int(len(validation_scores)),
    'anomaly_test_frames': int(len(test_scores)),
}
display(pd.DataFrame([summary]))

In [ ]:
OUTPUT_DIR = Path('/content/E04_results')
OUTPUT_DIR.mkdir(exist_ok=True)

normal_examples = validation_scores.nlargest(3, 'anomaly_score')
anomaly_examples = test_scores.nlargest(3, 'anomaly_score')
examples = pd.concat([normal_examples, anomaly_examples], ignore_index=True)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for axis, row in zip(axes.ravel(), examples.itertuples(index=False)):
    image = Image.open(DATA_ROOT / row.image_path).convert('RGB')
    heatmap = all_maps[(str(row.video), int(row.frame))]
    heatmap_image = Image.fromarray(heatmap.astype(np.float32), mode='F').resize(
        image.size, resample=Image.Resampling.BILINEAR
    )
    heatmap_image = np.asarray(heatmap_image)
    axis.imshow(image, cmap='gray')
    axis.imshow(heatmap_image, cmap='jet', alpha=0.45)

    frame_boxes = boxes_by_frame.get((str(row.video), int(row.frame)))
    if frame_boxes is not None:
        for box in frame_boxes.itertuples(index=False):
            axis.add_patch(patches.Rectangle(
                (box.xtl, box.ytl), box.xbr - box.xtl, box.ybr - box.ytl,
                fill=False, edgecolor='lime', linewidth=1.5,
            ))
    axis.set_title(f'{row.label} — score {row.anomaly_score:.3f}')
    axis.axis('off')

fig.suptitle('E04 — heatmap one-class; box CVAT in verde', fontsize=16)
fig.tight_layout()
figure_path = OUTPUT_DIR / 'qualitative_heatmaps.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
import shutil

scores.to_csv(OUTPUT_DIR / 'frame_scores.csv', index=False)
with (OUTPUT_DIR / 'summary.json').open('w') as handle:
    json.dump(summary, handle, indent=2)

ordered_maps = [all_maps[(str(row.video), int(row.frame))] for row in scores.itertuples(index=False)]
np.savez_compressed(
    OUTPUT_DIR / 'anomaly_maps.npz',
    maps=np.stack(ordered_maps),
    video=scores.video.to_numpy(),
    frame=scores.frame.to_numpy(),
    image_path=scores.image_path.to_numpy(),
)
shutil.copy('/content/E04_normal_memory_fp16.pt', OUTPUT_DIR / 'normal_memory_fp16.pt')

runtime = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'torchvision': __import__('torchvision').__version__,
    'gpu': torch.cuda.get_device_name(0),
    'seed': SEED,
    'backbone': 'ResNet18 ImageNet, frozen through layer3',
    'maximum_memory_patches': maximum_memory_patches,
}
with (OUTPUT_DIR / 'runtime.json').open('w') as handle:
    json.dump(runtime, handle, indent=2)

zip_path = shutil.make_archive('/content/E04_one_class_results', 'zip', OUTPUT_DIR)
print('Archivio creato:', zip_path)
files.download(zip_path)